In [38]:

import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from solarrpy.calibration import calibrate_window
from solarrpy.sorad import sorad_price, daily_strike
from solarrpy.buyer import unhedged_cf, benchmark_cf, hedged_cf, optimal_Nc, mape, tick

In [39]:
df = pd.read_csv('../data/CAMS_data/CAMS_data_Bologna.csv', parse_dates=['date'])
df = df.drop(columns=['H0']).copy()  # H0 is computed internally, and the provided values are not trustworthy

# Bologna Coordinates used in the paper
LATITUDE = 44.4949
LONGITUDE = 11.3426
ALTITUDE = 71

df_train = df[df['Year'] <= 2013].copy()
cal = calibrate_window(
    df_train,
    target_col='GHI', clearsky_col='clearsky', date_col='date',
    coords={'lat': LATITUDE}, train_end_year=2013,
)
print(f'theta={cal.theta:.4f}, alpha={cal.alpha:.4f}, beta={cal.beta:.4f}')

theta=0.5000, alpha=0.0100, beta=1.5000


# Asian SORAD

In [ ]:
# =============================================================================
# Asian SoRad (ASoRad) – Monthly average put hedging (Monte Carlo)
# Compatibility path: works with either the newer CalibrationWindowResult
# or the legacy CalibrationResult already present in the notebook kernel.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from solarrpy.buyer import unhedged_cf, benchmark_cf, mape, tick
from solarrpy.forecastDensity import clearsky_at, seasonal_mean_Y_at, R_to_Y, Y_to_R
from solarrpy.sorad import daily_strike

# -----------------------------------------------------------------------------
# Helpers: choose the modern calibration path when available, otherwise fall
# back to the legacy notebook variables/legacy CalibrationResult fields.
# -----------------------------------------------------------------------------
def _norm_date(date):
    return pd.Timestamp(date).normalize()

def _legacy_lookup(values, month, default=None):
    if isinstance(values, dict):
        return values.get(month, default)
    if isinstance(values, pd.Series):
        if month in values.index:
            return values.loc[month]
        if (month - 1) in values.index:
            return values.loc[month - 1]
    return default

def clearsky_C(date, cal):
    if hasattr(cal, 'clearsky_model'):
        return float(clearsky_at(date, cal))
    date = _norm_date(date)
    if 'clearsky' in df.columns:
        row = df.loc[df['date'].dt.normalize() == date, 'clearsky']
        if len(row):
            return float(row.iloc[0])
    if 'C_series' in globals():
        series = C_series
        if isinstance(series.index, pd.DatetimeIndex):
            try:
                return float(series.loc[date])
            except Exception:
                pass
    raise AttributeError('No clear-sky source available for the current calibration object')

def seasonal_mean_Y(date, cal):
    if hasattr(cal, 'a'):
        return float(seasonal_mean_Y_at(date, cal))
    date = _norm_date(date)
    month = int(date.month)
    value = _legacy_lookup(getattr(cal, 'seasonal_mean', None), month)
    if value is None and 'monthly_means' in globals():
        value = _legacy_lookup(monthly_means, month)
    if value is None:
        raise AttributeError('No seasonal mean source available for the current calibration object')
    return float(value)

def seasonal_sigma(date, cal):
    if hasattr(cal, 'c'):
        omega = 2 * np.pi / 365
        t = _norm_date(date).dayofyear
        return float(np.sqrt(cal.c[0] + cal.c[1] * np.sin(omega * t) + cal.c[2] * np.cos(omega * t)))
    date = _norm_date(date)
    month = int(date.month)
    variance = _legacy_lookup(getattr(cal, 'seasonal_variance', None), month)
    if variance is None and 'monthly_vars' in globals():
        variance = _legacy_lookup(monthly_vars, month)
    if variance is None:
        return float(getattr(cal, 'sigma', 1.0))
    return float(np.sqrt(max(float(variance), 0.0)))

def transform(R, date, cal):
    C = clearsky_C(date, cal)
    inside = 1 - cal.alpha - R / C
    if inside <= 0:
        inside = 1e-8
    return np.log(np.log(cal.beta) - np.log(inside))

def inverse_transform(Y, date, cal):
    C = clearsky_C(date, cal)
    return C * (1 - cal.alpha - cal.beta * np.exp(-Y))

# -----------------------------------------------------------------------------
# 1. Precompute daily values for 2014
# -----------------------------------------------------------------------------
n_days = 365
dates_2014 = pd.date_range('2014-01-01', periods=n_days, freq='D')
t = np.arange(1, n_days + 1)
omega = 2 * np.pi / 365

# Clear-sky radiation C_t
C_t = np.array([clearsky_C(d, cal) for d in dates_2014])

# Seasonal mean of Y (Ȳ_t) and seasonal volatility σ̄_t
Ybar_t = np.array([seasonal_mean_Y(d, cal) for d in dates_2014])
sigma_bar_t = np.array([seasonal_sigma(d, cal) for d in dates_2014])

# Monthly mixture parameters: use the full regime-switching version when
# available; otherwise fall back to a Gaussian monthly approximation from
# the legacy CalibrationResult.
monthly_df = cal.monthly_mixture.set_index('Month')
use_regime_mixture = {'p', 'mu0', 'mu1', 'sd0', 'sd1'}.issubset(set(monthly_df.columns))
if use_regime_mixture:
    p_month = monthly_df['p'].values
    mu0_month = monthly_df['mu0'].values
    mu1_month = monthly_df['mu1'].values
    sigma0_month = monthly_df['sd0'].values
    sigma1_month = monthly_df['sd1'].values
else:
    monthly_mean = monthly_df['mean'].values
    monthly_sd = np.sqrt(monthly_df['variance'].values)

# Initial Y on 2013-12-31
prev_date = pd.Timestamp('2013-12-31')
prev_R = df[df['date'] == prev_date]['GHI'].values[0]
Y_prev = transform(prev_R, prev_date, cal)

# -----------------------------------------------------------------------------
# 2. Monte Carlo simulation of daily GHI for 2014
# -----------------------------------------------------------------------------
np.random.seed(12345)
n_sim = 50000   # reduce to 10000 for a faster test run

def simulate_one_path():
    Y = np.zeros(n_days)
    Y_curr = Y_prev
    for i in range(n_days):
        if use_regime_mixture:
            month_idx = dates_2014[i].month - 1
            B = np.random.binomial(1, p_month[month_idx])        # 1 = cloudy
            mu_B = mu1_month[month_idx] if B == 1 else mu0_month[month_idx]
            sigma_B = sigma1_month[month_idx] if B == 1 else sigma0_month[month_idx]
            dY = (cal.theta * (Ybar_t[i] - Y_curr) + sigma_bar_t[i] * mu_B) + sigma_bar_t[i] * sigma_B * np.random.normal(0, 1)
        else:
            month_idx = dates_2014[i].month - 1
            dY = cal.theta * (Ybar_t[i] - Y_curr) + sigma_bar_t[i] * np.random.normal(0, 1)
        Y[i] = Y_curr + dY
        Y_curr = Y[i]
    R = np.array([inverse_transform(y, dates_2014[i], cal) for i, y in enumerate(Y)])
    return R

print(f"Simulating {n_sim} paths for 2014...")
all_R = np.zeros((n_sim, n_days))
for sim in range(n_sim):
    if sim % 10000 == 0:
        print(f"  Path {sim}/{n_sim}")
    all_R[sim] = simulate_one_path()
print("Simulation done.")

# -----------------------------------------------------------------------------
# 3. Daily strikes and actual GHI for 2014
# -----------------------------------------------------------------------------
df_2014 = df[df['Year'] == 2014].sort_values('date').reset_index(drop=True)
R_actual = df_2014['GHI'].values
days = np.arange(1, n_days + 1)
if hasattr(cal, 'clearsky_model') and hasattr(cal, 'a'):
    K_daily = np.array([daily_strike(row['date'], cal) for _, row in df_2014.iterrows()])
else:
    K_daily = np.array([clearsky_C(d, cal) * (1 - cal.alpha - cal.beta * np.exp(-np.exp(seasonal_mean_Y(d, cal)))) for d in df_2014['date']])

# -----------------------------------------------------------------------------
# 4. Price monthly Asian puts (put on monthly average radiation)
# -----------------------------------------------------------------------------
month_starts = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
month_ends   = [31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334, 365]
months = list(zip(month_starts, month_ends))

monthly_strikes = []
monthly_prices = []

for (s, e) in months:
    days_in_month = np.arange(s, e)
    K_month = np.mean(K_daily[days_in_month])
    monthly_strikes.append(K_month)
    avg_R_per_path = np.mean(all_R[:, days_in_month], axis=1)
    payoff_per_path = tick * np.maximum(K_month - avg_R_per_path, 0)
    monthly_prices.append(np.mean(payoff_per_path))

# Actual monthly payoffs using historical 2014 GHI
actual_monthly_payoff = []
for idx, (s, e) in enumerate(months):
    avg_R = np.mean(R_actual[s:e])
    payoff = tick * max(monthly_strikes[idx] - avg_R, 0)
    actual_monthly_payoff.append(payoff)

# -----------------------------------------------------------------------------
# 5. Distribute monthly premium and payoff equally over days in each month
# -----------------------------------------------------------------------------
daily_premium = np.zeros(n_days)
daily_payoff  = np.zeros(n_days)
for idx, ((s, e), prem) in enumerate(zip(months, monthly_prices)):
    days_in_month = np.arange(s, e)
    daily_premium[days_in_month] = prem / len(days_in_month)
    daily_payoff[days_in_month] = actual_monthly_payoff[idx] / len(days_in_month)

# -----------------------------------------------------------------------------
# 6. Optimal number of contracts (global Nc)
# -----------------------------------------------------------------------------
cf_u = unhedged_cf(R_actual)
cf_bench = benchmark_cf(K_daily)
diff = cf_u - cf_bench
delta = daily_payoff - daily_premium
Nc_opt = -np.sum(diff * delta) / np.sum(delta**2)
if Nc_opt < 0:
    Nc_opt = 0
print(f"Optimal number of Asian contracts (per month): {Nc_opt:.1f}")

# -----------------------------------------------------------------------------
# 7. Hedged cash flows and performance metrics
# -----------------------------------------------------------------------------
cf_hedged = cf_u + Nc_opt * delta
mape_u = mape(cf_u, cf_bench)
mape_h = mape(cf_hedged, cf_bench)
var_u = np.var(cf_u - cf_bench)
var_h = np.var(cf_hedged - cf_bench)

print("\n=== Asian SoRad Results ===")
print(f"MAPE unhedged = {mape_u:.2f}%")
print(f"MAPE hedged   = {mape_h:.2f}%")
print(f"Variance reduction: {100*(1 - var_h/var_u):.1f}%")

# -----------------------------------------------------------------------------
# 8. Plot (same style as Figure 4)
# -----------------------------------------------------------------------------
above = R_actual >= K_daily
legend_patches = [
    mpatches.Patch(color='#2ecc71', label='R_n >= K_n (above seasonal mean)'),
    mpatches.Patch(color='#e74c3c', label='R_n < K_n  (below seasonal mean)'),
]

# Dense benchmark for smooth curve
dates_dense = pd.date_range('2014-01-01', '2014-12-31', freq='D')
if hasattr(cal, 'clearsky_model') and hasattr(cal, 'a'):
    K_dense = np.array([daily_strike(d, cal) for d in dates_dense])
else:
    K_dense = np.array([clearsky_C(d, cal) * (1 - cal.alpha - cal.beta * np.exp(-np.exp(seasonal_mean_Y(d, cal)))) for d in dates_dense])
cfb_dense = benchmark_cf(K_dense)
days_dense = np.arange(1, len(dates_dense) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, cf_vals, title in [(axes[0], cf_u, 'Unhedged'),
                           (axes[1], cf_hedged, 'Hedged with Asian SoRad')]:
    ax.scatter(days[above],  cf_vals[above],  s=18, color='#2ecc71', alpha=0.85, zorder=3, linewidths=0)
    ax.scatter(days[~above], cf_vals[~above], s=18, color='#e74c3c', alpha=0.85, zorder=3, linewidths=0)
    ax.plot(days_dense, cfb_dense, 'k-', lw=1.6, zorder=4)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Day of the year', fontsize=10)
    ax.set_ylabel('Cash Flows (Eur)', fontsize=10)
    ax.set_xlim(0, 365)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.25)

fig.legend(handles=legend_patches, loc='lower center', ncol=2,
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.04))
fig.suptitle(
    f'Asian SoRad (monthly average put) – Buyer cash flows, Bologna 2014\n'
    f'Nc = {Nc_opt:.0f} | MAPE: {mape_u:.1f}% → {mape_h:.1f}%',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig('../results/Figure_ASoRad_buyer_cashflows.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved as ../results/Figure_ASoRad_buyer_cashflows.png")

# -----------------------------------------------------------------------------
# 9. Gate checks
# -----------------------------------------------------------------------------
assert cf_hedged.min() >= 0, "Negative hedged cash flow"
assert mape_h < mape_u, "Hedge did not reduce MAPE"
assert var_h < var_u, "Hedge did not reduce variance"
print("All gate checks passed.")

AttributeError: 'CalibrationResult' object has no attribute 'delta'

In [ ]:
# Compatibility smoke test for the legacy calibration object currently in the kernel.
def _norm_date(date):
    return pd.Timestamp(date).normalize()


def _legacy_lookup(values, month, default=None):
    if isinstance(values, dict):
        return values.get(month, default)
    if isinstance(values, pd.Series):
        if month in values.index:
            return values.loc[month]
        if (month - 1) in values.index:
            return values.loc[month - 1]
    return default


def _smoke_clearsky(date):
    date = _norm_date(date)
    row = df.loc[df['date'].dt.normalize() == date, 'clearsky']
    if len(row):
        return float(row.iloc[0])
    if 'C_series' in globals() and isinstance(C_series.index, pd.DatetimeIndex):
        return float(C_series.loc[date])
    raise AttributeError('No clear-sky source available')


def _smoke_seasonal_mean(date):
    date = _norm_date(date)
    month = int(date.month)
    value = _legacy_lookup(getattr(cal, 'seasonal_mean', None), month)
    if value is None and 'monthly_means' in globals():
        value = _legacy_lookup(monthly_means, month)
    if value is None:
        raise AttributeError('No seasonal mean source available')
    return float(value)


def _smoke_seasonal_sigma(date):
    date = _norm_date(date)
    month = int(date.month)
    variance = _legacy_lookup(getattr(cal, 'seasonal_variance', None), month)
    if variance is None and 'monthly_vars' in globals():
        variance = _legacy_lookup(monthly_vars, month)
    if variance is None:
        return float(getattr(cal, 'sigma', 1.0))
    return float(np.sqrt(max(float(variance), 0.0)))


d0 = pd.Timestamp('2014-01-01')
print(type(cal).__name__)
print(round(_smoke_clearsky(d0), 4), round(_smoke_seasonal_mean(d0), 4), round(_smoke_seasonal_sigma(d0), 4))
